# StrongReject Result Compilation

This notebook compiles the workflow-traced StrongReject experiment caches and displays the main summary, coverage, and reliability tables.

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import matplotlib.pyplot as plt

EXPECTED_PYTHON = Path("/workspace/.venv/bin/python")
if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError(
        "Wrong notebook kernel.\n"
        f"  got:      {sys.executable}\n"
        f"  expected: {EXPECTED_PYTHON}\n"
        "In Cursor: Select Kernel -> Python Environments -> Enter interpreter path:\n"
        "  /workspace/.venv/bin/python\n"
        "Or run from terminal: bash notebooks/run_compile_notebook.sh"
    )

def _resolve_extension_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'compile_strongreject_results.py').exists():
            return candidate
        nested = candidate / 'activation_oracles_extensions' / 'compile_strongreject_results.py'
        if nested.exists():
            return nested.parent
    raise FileNotFoundError('Could not locate compile_strongreject_results.py from current notebook cwd')

EXT_ROOT = _resolve_extension_root(Path.cwd())
os.chdir(EXT_ROOT)
sys.path.insert(0, str(EXT_ROOT))

from compile_strongreject_results import StrongRejectCompileConfig, compile_strongreject_results


## Configure And Compile

In [2]:
cfg = StrongRejectCompileConfig(
    cache_root=EXT_ROOT / 'cache',
    output_dir=EXT_ROOT / 'compiled_strongreject_results',
    judge_instruction_path='strongReject_v5.jinja2',
    expected_target_prompts=100,
    expected_target_rollouts=50,
    expected_oracle_rollouts=50,
)

manifest = compile_strongreject_results(cfg)
manifest['outputs']


ModuleNotFoundError: No module named 'datasets'

## Coverage

In [ ]:
coverage = pd.DataFrame([
    {
        'condition': condition,
        'expected_files': manifest.get('expected_files', {}).get(condition, 0),
        'loaded_files': manifest.get('loaded_files', {}).get(condition, 0),
    }
    for condition in sorted(set(manifest.get('expected_files', {})) | set(manifest.get('loaded_files', {})))
])
coverage['missing_files'] = coverage['expected_files'] - coverage['loaded_files']
coverage


In [ ]:
print(f"Missing expected files: {len(manifest.get('missing_files', []))}")
print(f"Malformed files: {len(manifest.get('malformed_files', []))}")
print(f"Skipped score leaves: {len(manifest.get('skipped_score_leaves', []))}")
pd.DataFrame(manifest.get('coverage_warnings', [])).head(20)


## Main Summary

In [ ]:
summary = pd.read_csv(cfg.output_dir / 'strongreject_summary.csv')
prompt_level = pd.read_csv(cfg.output_dir / 'strongreject_prompt_level.csv')
reliability = pd.read_csv(cfg.output_dir / 'strongreject_reliability.csv')

display_cols = [
    'condition', 'probe_name', 'oracle_prompt_file', 'n_prompts',
    'mean_score', 'se_score', 'asr_0_2', 'asr_0_5', 'asr_0_8', 'asr_1'
]
summary[display_cols].sort_values(['condition', 'oracle_prompt_file', 'probe_name']).head(100)


## Baselines

In [ ]:
summary.loc[summary['condition'].isin(['target_baseline', 'oracle_rollout_control']), display_cols]


## User Prompt Oracle

In [ ]:
user_prompt = summary[summary['condition'] == 'user_prompt_oracle']
user_prompt[display_cols].sort_values(['oracle_prompt_file', 'probe_name'])


## Target Rollout Oracle

In [ ]:
target_rollout = summary[summary['condition'] == 'target_rollout_oracle']
target_rollout[display_cols].sort_values(['oracle_prompt_file', 'probe_name'])


## Reliability

In [ ]:
reliability.sort_values(['condition', 'oracle_prompt_file', 'probe_name']).head(100)


## Quick Plot

In [ ]:
plot_df = summary.dropna(subset=['mean_score']).copy()
plot_df['label'] = plot_df['condition'] + ' | ' + plot_df['probe_name'].astype(str) + ' | ' + plot_df['oracle_prompt_file'].fillna('')
plot_df = plot_df.sort_values('mean_score', ascending=False).head(30)

ax = plot_df.plot.barh(x='label', y='mean_score', xerr='se_score', figsize=(10, 10), legend=False)
sample_size = f"actual prompts={manifest.get('actual_target_prompts')} / expected={manifest.get('expected_target_prompts')}"
ax.set_title(f'StrongReject summary ({sample_size})')
ax.set_xlabel('StrongReject score')
ax.set_ylabel('')
ax.invert_yaxis()
plt.tight_layout()
